# Chương 3 — EDA chuyên sâu dữ liệu ảnh: CIFAR-10

## Mục tiêu và nguồn dữ liệu

CIFAR-10 gồm ảnh RGB 32×32, 10 lớp đối tượng; split chuẩn có 50.000 ảnh train và 10.000 ảnh test. Nguồn tải: [University of Toronto CIFAR-10 trên Hugging Face](https://huggingface.co/datasets/uoft-cs/cifar10), thông tin gốc tại [trang CIFAR-10 của University of Toronto](https://www.cs.toronto.edu/~kriz/cifar.html).

Notebook chỉ dùng tập train cho EDA/pHash. Tập test được giữ riêng để đánh giá mô hình ở chương tiền xử lý. Phân tích ảnh trên mẫu có seed cố định khi việc tính toán toàn tập quá tốn tài nguyên; kích thước mẫu luôn được in rõ.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p for p in _candidates if (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy src/. Hãy mở notebook trong repository.")
sys.path.insert(0, str(REPO_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loading import load_cifar10_dataset
from src.image_analysis import compute_phashes, exact_duplicate_groups, phash_collision_groups, near_duplicate_pairs, remove_exact_duplicate_indices
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
print("Repository:", REPO_ROOT)

## 1. Tải dữ liệu, xác thực quy mô và kiểm tra ảnh mẫu

Dữ liệu được tải từ Hugging Face ở lần chạy đầu. Mảng ảnh dùng `uint8` để tiết kiệm RAM (khoảng 150 MB cho 50.000 ảnh train 32×32 RGB). Không ghi mảng ảnh vào Git.

In [ ]:
X_train, y_train, X_test, y_test, class_names = load_cifar10_dataset()
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Classes:", class_names)
validation = {
    "total images >= 5,000": len(X_train) + len(X_test) >= 5_000,
    "classes >= 5": len(np.unique(np.concatenate([y_train, y_test]))) >= 5,
    "RGB image tensor": X_train.ndim == 4 and X_train.shape[-1] == 3,
}
display(pd.DataFrame({"requirement": validation.keys(), "passed": validation.values()}))
assert all(validation.values()), "CIFAR-10 hiện không đạt ngưỡng đề bài."

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for class_id, ax in enumerate(axes.flat):
    idx = np.flatnonzero(y_train == class_id)[0]
    ax.imshow(X_train[idx])
    ax.set_title(class_names[class_id])
    ax.axis("off")
plt.suptitle("One training image from each CIFAR-10 class")
plt.tight_layout(); plt.show()

## 2. Phân tích mất cân bằng lớp

Tính số ảnh, tỷ lệ từng lớp và tỷ số giữa lớp nhiều nhất với lớp ít nhất. Theo đề, cần kiểm tra liệu tỷ lệ có vượt mức 3 lần hay không.

In [ ]:
class_counts = pd.Series(y_train).value_counts().sort_index()
class_table = pd.DataFrame({
    "class_name": [class_names[i] for i in class_counts.index],
    "image_count": class_counts.values,
    "share": class_counts.values / len(y_train),
})
imbalance_ratio = class_counts.max() / class_counts.min()
display(class_table.style.format({"share": "{:.2%}"}))
print(f"Max/min class ratio = {imbalance_ratio:.3f}")
print("Có lớp vượt 3 lần lớp ít nhất:", bool(imbalance_ratio > 3))
plt.figure(figsize=(10, 4))
sns.barplot(data=class_table, x="class_name", y="image_count")
plt.xticks(rotation=35, ha="right"); plt.title("CIFAR-10 training class distribution")
plt.tight_layout(); plt.show()

## 3. Histogram và KDE giá trị pixel theo kênh màu

CIFAR-10 là ảnh RGB. Lấy mẫu tối đa 300.000 pixel để vẽ histogram/KDE; thống kê pixel không phụ thuộc nhãn và mẫu được chọn ngẫu nhiên có seed cố định. Trục x thuộc [0,255].

In [ ]:
all_pixels = X_train.reshape(-1, 3)
pixel_sample_n = min(300_000, len(all_pixels))
pixel_idx = rng.choice(len(all_pixels), size=pixel_sample_n, replace=False)
pixel_sample = all_pixels[pixel_idx]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for channel, ax, channel_name in zip(range(3), axes, ["Red", "Green", "Blue"]):
    sns.histplot(pixel_sample[:, channel], bins=64, stat="density", kde=True, ax=ax)
    ax.set_title(f"{channel_name} pixel distribution")
    ax.set_xlabel("Pixel intensity")
plt.tight_layout(); plt.show()
print(f"Số pixel dùng vẽ mỗi kênh: {pixel_sample_n:,}")

## 4. Độ sáng và độ tương phản theo lớp

Mean intensity đại diện độ sáng toàn cục; standard deviation (std) của cường độ xám phản ánh mức biến thiên/độ tương phản. Đây là đặc trưng mô tả thô, không thay thế đánh giá thị giác hoặc phân tích cấu trúc ảnh.

In [ ]:
# Per-image grayscale summary; process in batches to limit intermediate memory.
intensity_means = np.empty(len(X_train), dtype=np.float32)
intensity_stds = np.empty(len(X_train), dtype=np.float32)
for start in range(0, len(X_train), 2_000):
    stop = min(start + 2_000, len(X_train))
    batch = X_train[start:stop].astype(np.float32) / 255.0
    gray = 0.2126 * batch[..., 0] + 0.7152 * batch[..., 1] + 0.0722 * batch[..., 2]
    intensity_means[start:stop] = gray.mean(axis=(1, 2))
    intensity_stds[start:stop] = gray.std(axis=(1, 2))

intensity_df = pd.DataFrame({
    "class_name": [class_names[i] for i in y_train],
    "mean_intensity": intensity_means,
    "intensity_std": intensity_stds,
})
intensity_summary = intensity_df.groupby("class_name").agg(
    mean_intensity_mean=("mean_intensity", "mean"),
    mean_intensity_std=("mean_intensity", "std"),
    contrast_mean=("intensity_std", "mean"),
    contrast_std=("intensity_std", "std"),
)
display(intensity_summary)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=intensity_df, x="class_name", y="mean_intensity", ax=axes[0])
axes[0].set_title("Mean brightness by class"); axes[0].tick_params(axis="x", rotation=35)
sns.boxplot(data=intensity_df, x="class_name", y="intensity_std", ax=axes[1])
axes[1].set_title("Grayscale intensity std by class"); axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

## 5. Phát hiện ảnh trùng/gần trùng bằng perceptual hash (pHash)

pHash nén cấu trúc tần số thấp của ảnh thành chuỗi 64 bit. Khoảng cách Hamming nhỏ cho biết hai hash gần nhau. Notebook:

1. Tính pHash 64-bit trên toàn bộ ảnh train; nhóm có cùng hash là ứng viên trùng, chưa đủ chứng minh pixel giống nhau.
2. Tìm near-duplicate với ngưỡng Hamming ≤ 3 trên mẫu tối đa 5.000 ảnh để tránh phép so sánh toàn cặp O(n²).
3. Xác minh ảnh giống hệt bằng SHA-256 trên mảng pixel; chỉ tự động loại bản sao pixel-identical. Các ứng viên pHash/near-duplicate khác được giữ để tránh false positive.

In [ ]:
hash_cache = REPO_ROOT / "data" / "processed" / "cifar10_train_phashes.npy"
if hash_cache.exists():
    phashes = np.load(hash_cache)
else:
    phashes = compute_phashes(X_train)
    hash_cache.parent.mkdir(parents=True, exist_ok=True)
    np.save(hash_cache, phashes)

# pHash equality means visual similarity/collision, not guaranteed pixel identity.
phash_groups = phash_collision_groups(phashes)
phash_collision_extra = sum(len(group) - 1 for group in phash_groups)
phash_collision_rate = phash_collision_extra / len(X_train)

# Confirm exact duplicates from raw pixel bytes before automatically removing any image.
exact_groups = exact_duplicate_groups(X_train)
exact_remove_indices = remove_exact_duplicate_indices(exact_groups)
exact_duplicate_count = len(exact_remove_indices)
exact_duplicate_rate = exact_duplicate_count / len(X_train)

print(f"pHash-equal candidate groups: {len(phash_groups)}")
print(f"Images beyond first member in pHash-equal groups: {phash_collision_extra:,} ({phash_collision_rate:.3%})")
print(f"Byte-identical image groups (SHA-256 pixels): {len(exact_groups)}")
print(f"Images beyond first exact copy: {exact_duplicate_count:,} ({exact_duplicate_rate:.3%})")

near_pairs = near_duplicate_pairs(phashes, max_images=5_000, max_hamming=3, random_state=RANDOM_SEED)
print(f"Near-duplicate candidate pairs in deterministic 5,000-image subset: {len(near_pairs):,}")
near_duplicate_table = pd.DataFrame(near_pairs, columns=["train_index_1", "train_index_2", "hamming_distance"])
display(near_duplicate_table.head(20))


In [ ]:
# Remove only byte-identical pixel arrays. pHash-near matches remain candidates for manual review.
keep_mask = np.ones(len(X_train), dtype=bool)
keep_mask[exact_remove_indices] = False
X_train_deduplicated = X_train[keep_mask]
y_train_deduplicated = y_train[keep_mask]
print("Original train images:", len(X_train))
print("After byte-identical duplicate removal:", len(X_train_deduplicated))
print("pHash/near-duplicate candidates are not auto-deleted because perceptual hashes can collide.")


### Kết luận EDA ảnh cần điền sau khi chạy

Báo cáo tỷ lệ mỗi lớp, tỷ số max/min, hình dạng phân phối pixel RGB, khác biệt độ sáng/độ tương phản theo lớp, số nhóm trùng hash pHash, số bản sao pixel-identical và số cặp near-duplicate trong mẫu. Nêu rõ giới hạn: near-duplicate chỉ rà soát trên 5.000 ảnh có seed cố định, không khẳng định toàn bộ 50.000 ảnh không có near-duplicate.